# 📩 Detector de Spam en SMS — versión en **español** (Bi-GRU)

Entrena el mismo modelo del proyecto, pero con mensajes en español. Usa el dataset **SMS Spam Multilingual Collection** (Hugging Face), que contiene la colección SMS Spam original traducida a 20+ idiomas.

> ⚠️ **Importante:** las traducciones al español fueron hechas con un traductor automático (M2M100), no son mensajes escritos por personas. El modelo funcionará bien con español "estándar", pero puede fallar con jerga, abreviaturas o modismos locales.

**Resultado final:** `models/spam_gru_model_es.keras` + `reports/metrics_es.json`, que se copian a la carpeta del proyecto para que la app los use.

## 0. Configuración

In [ ]:
!pip install -q langdetect
import os, random, json, re, unicodedata
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
MAX_TOKENS, SEQ_LEN, EMB_DIM, THRESHOLD = 8000, 50, 64, 0.5
os.makedirs("models", exist_ok=True); os.makedirs("reports", exist_ok=True)
print("TensorFlow", tf.__version__)

## 1. Selección y descripción del dataset

| | |
|---|---|
| **Nombre** | SMS Spam Multilingual Collection (`dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset`, Hugging Face) |
| **Origen** | SMS Spam Collection (Almeida & Hidalgo, UCI), traducida a 20+ idiomas, entre ellos español |
| **Tarea** | Clasificación binaria de texto: `spam` vs `ham` (legítimo) |
| **Por qué es secuencial** | Un SMS es una secuencia de palabras: el orden y el contexto importan, ideal para una GRU |
| **Reto** | Clases desbalanceadas (~13 % spam) → se evalúa con precisión, recall, F1 y AUC, no solo accuracy |
| **Limitación** | El español es traducción automática, no mensajes escritos por personas |

### Descarga

In [ ]:
CSV_PATH = "data-augmented.csv"
URL = "https://huggingface.co/datasets/dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset/resolve/main/data-augmented.csv"
if not os.path.exists(CSV_PATH):
    !wget -q -O {CSV_PATH} "{URL}"
raw = pd.read_csv(CSV_PATH)
print("Filas:", len(raw), "| Columnas:", raw.columns.tolist())
raw.head(10)

## 2. Detección de columnas

No asumo los nombres de las columnas: el código busca la de texto (la de mensajes más largos), la de etiqueta (`spam`/`ham` o 0/1) y, si existe, una de idioma. **Si algo sale mal, fija los nombres a mano** en las variables `TEXT_COL`, `LABEL_COL`, `LANG_COL`.

In [ ]:
TEXT_COL = LABEL_COL = LANG_COL = None   # <- rellena a mano si la detección falla, p. ej. TEXT_COL="text"

obj_cols = raw.select_dtypes(include=["object", "string"]).columns.tolist()
if TEXT_COL is None:
    TEXT_COL = max(obj_cols, key=lambda c: raw[c].astype(str).str.len().mean())
if LABEL_COL is None:
    ok = {"spam", "ham", "0", "1", "legitimate", "no spam", "not spam"}
    cands = [c for c in raw.columns if c != TEXT_COL and set(raw[c].dropna().astype(str).str.lower().unique()) <= ok]
    cands.sort(key=lambda c: raw[c].dtype not in ("object", "string"))   # prefiere la de texto (spam/ham)
    LABEL_COL = cands[0]
if LANG_COL is None:
    LANG_COL = next((c for c in raw.columns if "lang" in c.lower()), None)
print("TEXT_COL =", TEXT_COL, "| LABEL_COL =", LABEL_COL, "| LANG_COL =", LANG_COL)

lab = raw[LABEL_COL].astype(str).str.lower()
df = pd.DataFrame({"text": raw[TEXT_COL].astype(str), "y": lab.isin(["spam", "1"]).astype(int)})
if LANG_COL: df["lang"] = raw[LANG_COL].astype(str)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print("Filas únicas:", len(df)); print(df["y"].value_counts())

## 3. Filtrar solo los mensajes en español

Si el CSV trae columna de idioma se usa; si no, se detecta el idioma con `langdetect` (puede tardar unos minutos). Se exige confianza ≥ 90 % para evitar mezclar idiomas.

In [ ]:
from langdetect import detect_langs, DetectorFactory
DetectorFactory.seed = SEED

if LANG_COL:
    print(df["lang"].value_counts().head(25))
    es = df[df["lang"].str.lower().isin(["es", "spa", "spanish", "español", "espanol"])].copy()
else:
    def is_es(t):
        try:
            return any(l.lang == "es" and l.prob >= 0.90 for l in detect_langs(t))
        except Exception:
            return False
    try:
        from tqdm.auto import tqdm; tqdm.pandas()
        df["es"] = df["text"].progress_apply(is_es)
    except ImportError:
        df["es"] = df["text"].apply(is_es)
    es = df[df["es"]].copy()

es = es.reset_index(drop=True)
print(f"\nMensajes en español: {len(es)} | spam: {es['y'].sum()} ({es['y'].mean():.1%})")
assert len(es) > 1000, "Muy pocos mensajes en español: revisa LANG_COL / el filtro de idioma."
es.sample(8, random_state=1)[["text", "y"]]

## 4. EDA

In [ ]:
es["length"] = es["text"].str.len(); es["n_words"] = es["text"].str.split().str.len()
es["label"] = es["y"].map({0: "ham", 1: "spam"})
display(es.groupby("label")[["length", "n_words"]].describe().T.round(1))
pal = ["#4C72B0", "#C44E52"]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
es["label"].value_counts().plot.bar(ax=ax[0], color=pal); ax[0].set_title("Distribución de clases")
sns.histplot(data=es, x="length", hue="label", bins=40, ax=ax[1], palette=pal[::-1]); ax[1].set_title("Longitud (caracteres)")
sns.boxplot(data=es, x="label", y="n_words", hue="label", legend=False, ax=ax[2], palette=pal); ax[2].set_title("Nº de palabras")
plt.tight_layout(); plt.show()

## 5. Limpieza (conserva tildes y ñ)

In [ ]:
URL_RE, EMAIL_RE = re.compile(r"(https?://\S+|www\.\S+)"), re.compile(r"\S+@\S+")
PHONE_RE, NUM_RE, MONEY_RE = re.compile(r"\b\d{5,}\b"), re.compile(r"\b\d+\b"), re.compile(r"[£$€]")

def clean_text_es(text: str) -> str:   # idéntica a spam_utils.clean_text_es
    text = str(text).lower()
    text = URL_RE.sub(" urltoken ", text); text = EMAIL_RE.sub(" emailtoken ", text)
    text = MONEY_RE.sub(" moneytoken ", text); text = PHONE_RE.sub(" phonetoken ", text)
    text = NUM_RE.sub(" numtoken ", text)
    text = re.sub(r"[^a-záéíóúüñ\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

es["clean"] = es["text"].apply(clean_text_es)
es = es[es["clean"].str.len() > 0].reset_index(drop=True)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, lab_, col in zip(ax, ["ham", "spam"], pal):
    cnt = Counter(" ".join(es[es.label == lab_]["clean"]).split())
    w, c = zip(*cnt.most_common(15)); a.barh(w[::-1], c[::-1], color=col); a.set_title(f"Top 15 palabras — {lab_}")
plt.tight_layout(); plt.show()
es[["text", "clean"]].sample(5, random_state=2)

## 6. División 70 / 15 / 15 (estratificada)

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(es["clean"], es["y"], test_size=0.30, stratify=es["y"], random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
raw_test = es.loc[X_test.index, "text"].values
X_train, X_val, X_test = X_train.values, X_val.values, X_test.values
y_train, y_val, y_test = y_train.values, y_val.values, y_test.values
for n, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{n:5}: {len(y):5d} | % spam = {y.mean():.1%}")

## 7. Modelo Bi-GRU y entrenamiento

In [ ]:
vectorizer = layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN)
vectorizer.adapt(X_train)
inputs = keras.Input(shape=(), dtype="string")
x = vectorizer(inputs)
x = layers.Embedding(MAX_TOKENS, EMB_DIM, mask_zero=True)(x)
x = layers.Bidirectional(layers.GRU(32))(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(32, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="spam_bigru_es")
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
model.summary()

In [ ]:
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: cw[0], 1: cw[1]}
def make_ds(X, y=None, shuffle=False):
    t = tf.constant(X, dtype=tf.string)
    ds = tf.data.Dataset.from_tensor_slices((t, y.astype("float32")) if y is not None else t)
    if shuffle: ds = ds.shuffle(len(X), seed=SEED)
    return ds.batch(32).prefetch(1)
es_cb = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
history = model.fit(make_ds(X_train, y_train, True), validation_data=make_ds(X_val, y_val),
                    epochs=30, class_weight=class_weight, callbacks=[es_cb], verbose=2)

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, m, t in zip(ax, ["loss", "accuracy", "recall"], ["Pérdida", "Exactitud", "Recall"]):
    a.plot(history.history[m], label="train"); a.plot(history.history[f"val_{m}"], label="val"); a.set_title(t); a.legend()
plt.tight_layout(); plt.show()

## 8. Evaluación en test

In [ ]:
proba = model.predict(make_ds(X_test), verbose=0).ravel()
pred = (proba >= THRESHOLD).astype(int)
metrics = {"accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
           "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred), "roc_auc": roc_auc_score(y_test, proba)}
print(pd.Series(metrics).round(4).to_string()); print()
print(classification_report(y_test, pred, target_names=["ham", "spam"]))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", ax=ax[0], xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
ax[0].set_title("Matriz de confusión"); ax[0].set_xlabel("Predicho"); ax[0].set_ylabel("Real")
fpr, tpr, _ = roc_curve(y_test, proba); ax[1].plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}"); ax[1].plot([0, 1], [0, 1], "--")
ax[1].set_title("Curva ROC"); ax[1].legend(); plt.tight_layout(); plt.show()

## 9. Predicciones de prueba

In [ ]:
res = pd.DataFrame({"mensaje": raw_test, "real": np.where(y_test == 1, "spam", "ham"),
                    "prob_spam": proba.round(4), "predicho": np.where(pred == 1, "spam", "ham")})
res["correcto"] = np.where(res["real"] == res["predicho"], "✅", "❌")
muestra = pd.concat([res[res.real == "spam"].sample(8, random_state=SEED), res[res.real == "ham"].sample(8, random_state=SEED)])
pd.set_option("display.max_colwidth", 90); display(muestra.reset_index(drop=True))

nuevos = [
    "¡Felicidades! Has ganado un premio de 1000 euros. Llama al 09061701461 ahora para reclamarlo.",
    "Hola, ¿sigues queriendo almorzar mañana? Avísame a qué hora te viene bien.",
    "URGENTE: tu cuenta ha sido suspendida. Verifica tus datos en www.banco-seguro-login.com",
    "No olvides la reunión de mañana a las 10, llevaré los documentos.",
    "Entrada gratis a nuestro sorteo semanal. Envía GANA al 80086 para recibir tu tono.",
]
p = model.predict(tf.constant([clean_text_es(t) for t in nuevos]), verbose=0).ravel()
for t, pi in zip(nuevos, p): print(f"[{'SPAM' if pi >= THRESHOLD else 'HAM ':4}] P(spam)={pi:.3f} | {t}")

## 10. Guardado

In [ ]:
model.save("models/spam_gru_model_es.keras")
json.dump({k: round(float(v), 4) for k, v in metrics.items()}, open("reports/metrics_es.json", "w"), indent=2)
reloaded = keras.models.load_model("models/spam_gru_model_es.keras")
print("Recarga idéntica:", np.allclose(p, reloaded.predict(tf.constant([clean_text_es(t) for t in nuevos]), verbose=0).ravel(), atol=1e-5))

# En Colab, descarga los dos archivos:
# from google.colab import files
# files.download("models/spam_gru_model_es.keras"); files.download("reports/metrics_es.json")

## 11. Siguiente paso
Copia **`spam_gru_model_es.keras`** a `spam_project/models/` y **`metrics_es.json`** a `spam_project/reports/`. Al ejecutar `streamlit run app.py` aparecerá el selector **Español** en la barra lateral.